<a href="https://colab.research.google.com/github/mounikas-tech-AI/Deeplearning_Projects/blob/main/Represent_facial_expression_in_emoji.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Real-Time Emotion and Emoji Recognition System

This notebook demonstrates a complete end-to-end pipeline to train a Convolutional Neural Network (CNN) to detect human emotions from grayscale facial images and map them to their corresponding emojis using OpenCV and Tkinter interfaces.

---

## 🛠️ Project Architecture
1. **Data Preparation**: Loading grayscaled face dataset (such as FER2013) from local directories using `ImageDataGenerator` with scaling normalization.
2. **Model Design**: Building a deep 2D Convolutional Neural Network using Keras Sequential API.
3. **Model Training**: Compiling and training the model on CPU/GPU, saving the weights locally as `emotion_model.h5`.
4. **Real-time Deployment**:
    * **Webcam feed**: Live OpenCV facial bounding boxes and emotion projection.
    * **Tkinter GUI Application**: Interactive display tracking human expressions with corresponding emojis.

---

## 🗂️ Setup and Requirements
Ensure you have the following folder structure for your scripts and assets:
```text
├── Documents/
│   └── Python Scripts/
│       ├── train/            # Train folder (7 classes of emotions)
│       └── test/             # Test/Val folder (7 classes of emotions)
└── emoji/
    ├── haarcascade_frontalface_default.xml
    ├── indianservers.jpg
    ├── angry.png
    ├── disgusted.png
    ├── fearful.png
    ├── happy.png
    ├── neutral.png
    ├── sad.png
    └── surpriced.png
```
```

In [ ]:
import numpy as np
import argparse
import matplotlib.pyplot as plt
import cv2

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

```markdown
## 1. Data Preprocessing & Pipeline Initialization
In this step, we configure the directories containing our training and validation datasets, rescale the pixel values to the standard range of `[0, 1]`, and load the images as grayscale frames of size `48x48`.
```

In [ ]:
train_dir = 'Documents/Python Scripts/test'
val_dir = 'Documents/Python Scripts/train'

num_train = 28709
num_val = 7178
batch_size = 64
num_epoch = 5

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(48,48),
        batch_size=batch_size,
        color_mode="grayscale",
        class_mode='categorical')

validation_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(48,48),
        batch_size=batch_size,
        color_mode="grayscale",
        class_mode='categorical')

Found 7178 images belonging to 7 classes.
Found 28709 images belonging to 7 classes.


```markdown
## 2. Convolutional Neural Network (CNN) Architecture
We build a Keras `Sequential` model optimized for extraction of spatial local features in expressions:
* **Conv2D & MaxPooling**: Extracted deep patterns.
* **Dropout**: Prevent model overfitting.
* **Dense**: Map features into 7 classification output units with a `softmax` activation function.
```

In [ ]:
emotion_model = Sequential()

emotion_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48,48,1)))
emotion_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Flatten())
emotion_model.add(Dense(1024, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(7, activation='softmax'))

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


```markdown
## 3. Model Training & Compilation
We compile our CNN using the standard categorical cross-entropy loss function and the `Adam` optimizer. We track loss and accuracy progress across several training epochs.
```

In [ ]:
# emotion_model.load_weights('emotion_model.h5')

cv2.ocl.setUseOpenCL(False)


emotion_dict = {0: "Angry", 1: "Disgusted", 2: "Fearful", 3: "Happy", 4: "Neutral", 5: "Sad", 6: "Surprised"}


emotion_model.compile(loss='categorical_crossentropy',optimizer=Adam(lr=0.0001, decay=1e-6),metrics=['accuracy'])
emotion_model_info = emotion_model.fit_generator(
        train_generator,
        steps_per_epoch=28709 // 64,
        epochs=5,
        validation_data=validation_generator,
        validation_steps=7178 // 64)

Epoch 1/5
448/448 [==============================] - 1596s 4s/step - loss: 1.8101 - acc: 0.2555 - val_loss: 1.7375 - val_acc: 0.3175
Epoch 2/5
448/448 [==============================] - 1653s 4s/step - loss: 1.6290 - acc: 0.3674 - val_loss: 1.5561 - val_acc: 0.4035
Epoch 3/5
448/448 [==============================] - 1217s 3s/step - loss: 1.5040 - acc: 0.4229 - val_loss: 1.4704 - val_acc: 0.4353
Epoch 4/5
448/448 [==============================] - 1002s 2s/step - loss: 1.4043 - acc: 0.4655 - val_loss: 1.4143 - val_acc: 0.4595
Epoch 5/5
448/448 [==============================] - 952s 2s/step - loss: 1.3039 - acc: 0.5110 - val_loss: 1.3959 - val_acc: 0.4688


In [ ]:
emotion_model.save_weights('emotion_model.h5')

```markdown
## 4. Live OpenCV Face Detection and Emotion Inference
Here we load our local system camera feed (index 0) and evaluate frame-by-frame outputs using Viola-Jones Haar Cascades to crop out faces, predict the emotion, and project it overlaying the feed live.
```

In [ ]:
cv2.ocl.setUseOpenCL(False)
emotion_dict = {0: "Angry", 1: "Disgusted", 2: "Fearful", 3: "Happy", 4: "Neutral", 5: "Sad", 6: "Surprised"}
# start the webcam feed
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    bounding_box = cv2.CascadeClassifier('emoji/haarcascade_frontalface_default.xml')
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    num_faces = bounding_box.detectMultiScale(gray_frame,scaleFactor=1.3, minNeighbors=5)
    for (x, y, w, h) in num_faces:
        cv2.rectangle(frame, (x, y-50), (x+w, y+h+10), (255, 0, 0), 2)
        roi_gray_frame = gray_frame[y:y + h, x:x + w]
        cropped_img = np.expand_dims(np.expand_dims(cv2.resize(roi_gray_frame, (48, 48)), -1), 0)
        emotion_prediction = emotion_model.predict(cropped_img)
        maxindex = int(np.argmax(emotion_prediction))
        cv2.putText(frame, emotion_dict[maxindex], (x+20, y-60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imshow('Video', cv2.resize(frame,(1200,860),interpolation = cv2.INTER_CUBIC))
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()




KeyboardInterrupt: 

```markdown
## 5. Desktop Application: GUI Photo-to-Emoji Transformer
Using `tkinter`, we build an interactive GUI displaying the live camera on the left and updating the matching emoji asset on the right panel dynamically based on your current expression.
```

In [ ]:
import tkinter as tk
from tkinter import *
import cv2
from PIL import Image, ImageTk
import os
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
emotion_model = Sequential()
emotion_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48,48,1)))
emotion_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))
emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))
emotion_model.add(Flatten())
emotion_model.add(Dense(1024, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(7, activation='softmax'))
emotion_model.load_weights('emotion_model.h5')
cv2.ocl.setUseOpenCL(False)
emotion_dict = {0: "   Angry   ", 1: "Disgusted", 2: "  Fearful  ", 3: "   Happy   ", 4: "  Neutral  ", 5: "    Sad    ", 6: "Surprised"}
emoji_dist={0:"./emoji/angry.png",2:"./emoji/disgusted.png",2:"./emoji/fearful.png",3:"./emoji/happy.png",4:"./emoji/neutral.png",5:"./emoji/sad.png",6:"./emoji/surpriced.png"}
global last_frame1
last_frame1 = np.zeros((480, 640, 3), dtype=np.uint8)
global cap1
show_text=[0]
def show_vid():
    cap1 = cv2.VideoCapture(0)
    if not cap1.isOpened():
        print("cant open the camera1")
    flag1, frame1 = cap1.read()
    frame1 = cv2.resize(frame1,(600,500))
    bounding_box = cv2.CascadeClassifier('emoji/haarcascade_frontalface_default.xml')
    gray_frame = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    num_faces = bounding_box.detectMultiScale(gray_frame,scaleFactor=1.3, minNeighbors=5)
    for (x, y, w, h) in num_faces:
        cv2.rectangle(frame1, (x, y-50), (x+w, y+h+10), (255, 0, 0), 2)
        roi_gray_frame = gray_frame[y:y + h, x:x + w]
        cropped_img = np.expand_dims(np.expand_dims(cv2.resize(roi_gray_frame, (48, 48)), -1), 0)
        prediction = emotion_model.predict(cropped_img)

        maxindex = int(np.argmax(prediction))
        cv2.putText(frame1, emotion_dict[maxindex], (x+20, y-60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        show_text[0]=maxindex
    if flag1 is None:
        print ("Major error!")
    elif flag1:
        global last_frame1
        last_frame1 = frame1.copy()
        pic = cv2.cvtColor(last_frame1, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(pic)
        imgtk = ImageTk.PhotoImage(image=img)
        lmain.imgtk = imgtk
        lmain.configure(image=imgtk)
        lmain.after(10, show_vid)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        exit()
def show_vid2():
    frame2=cv2.imread(emoji_dist[show_text[0]])
    pic2=cv2.cvtColor(frame2,cv2.COLOR_BGR2RGB)
    img2=Image.fromarray(frame2)
    imgtk2=ImageTk.PhotoImage(image=img2)
    lmain2.imgtk2=imgtk2
    lmain3.configure(text=emotion_dict[show_text[0]],font=('arial',45,'bold'))

    lmain2.configure(image=imgtk2)
    lmain2.after(10, show_vid2)
if __name__ == '__main__':
    root=tk.Tk()
    img = ImageTk.PhotoImage(Image.open("emoji/indianservers.jpg"))
    heading = Label(root,image=img,bg='black')

    heading.pack()
    heading2=Label(root,text="Photo to Emoji",pady=20, font=('arial',45,'bold'),bg='black',fg='#CDCDCD')

    heading2.pack()
    lmain = tk.Label(master=root,padx=50,bd=10)
    lmain2 = tk.Label(master=root,bd=10)

    lmain3=tk.Label(master=root,bd=10,fg="#CDCDCD",bg='black')
    lmain.pack(side=LEFT)
    lmain.place(x=50,y=250)
    lmain3.pack()
    lmain3.place(x=960,y=250)
    lmain2.pack(side=RIGHT)
    lmain2.place(x=900,y=350)

    root.title("Photo To Emoji")
    root.geometry("1400x900+100+10")
    root['bg']='black'
    exitbutton = Button(root, text='Quit',fg="red",command=root.destroy,font=('arial',25,'bold')).pack(side = BOTTOM)
    show_vid()
    show_vid2()
    root.mainloop()


Exception ignored in: <function BaseSession._Callable.__del__ at 0x000001E8B9CD9048>
Traceback (most recent call last):
  File "C:\Users\LENOVO\Anaconda3\lib\site-packages\tensorflow\python\client\session.py", line 1471, in __del__
    if self._handle is not None and self._session._session is not None:
KeyboardInterrupt
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\LENOVO\Anaconda3\lib\tkinter\__init__.py", line 1705, in __call__
    return self.func(*args)
  File "C:\Users\LENOVO\Anaconda3\lib\tkinter\__init__.py", line 749, in callit
    func(*args)
  File "<ipython-input-9-59111dda545c>", line 40, in show_vid
    flag1, frame1 = cap1.read()
KeyboardInterrupt
